# 02 --- Analysis: Tiers 4-6 and report outputsReads `result.json` / `per_case.json` from Drive. **No training here**, so thisruns on CPU and costs nothing.Produces: statistics, volume stratification, post-processing ablation,Grad-CAM figures, and every LaTeX table.

## 2.0 --- Setup

In [ ]:
# 0.1 --- environment!pip -q install "monai==1.4.0" einops nibabel scipy scikit-image!nvidia-smi --query-gpu=name,memory.total --format=csvfrom google.colab import drive; drive.mount('/content/drive')import os, sys, subprocessREPO = '/content/499A'if not os.path.exists(REPO):    !git clone https://github.com/ahnaf-csg/499A---3d-Tumor-Segmentation.git {REPO}else:    subprocess.run(['git','-C',REPO,'pull'])sys.path.insert(0, REPO)import torch, monaiprint('torch', torch.__version__, '| monai', monai.__version__, '| cuda', torch.cuda.is_available())assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'cap = torch.cuda.get_device_capability(0)print(f'SM {cap[0]}.{cap[1]}  bf16={"yes" if cap[0]>=8 else "NO -> fp16 path (T4)"}')

In [ ]:
DRIVE = '/content/drive/MyDrive/Colab Notebooks/499a'OUT   = f'{DRIVE}/artifacts'TEX   = f'{DRIVE}/results/tex'import json, globfrom pathlib import Pathimport pandas as pdruns = {}for p in sorted(glob.glob(f'{OUT}/*/result.json')):    r = json.loads(Path(p).read_text())    r['_dir'] = str(Path(p).parent)    runs[Path(p).parent.name] = rprint(f'{len(runs)} completed runs found')pd.DataFrame(runs.values())[['name','model','mean_dice_mean','ET_dice_mean','seed']]

## 2.1 --- Statistics: does the difference exceed seed noise?Paired Wilcoxon on per-case Dice, Cohen's d, bootstrap CIs, FDR across comparisons.

In [ ]:
from glioseg.metrics import compare_arms, bootstrap_ci, fdr_correctdef per_case(dirname):    return json.loads((Path(runs[dirname]['_dir'])/'per_case.json').read_text())names = list(runs)print('available:', names)A, B = names[0], names[1]     # <-- set these to the two runs you want comparedcmp = compare_arms(per_case(A), per_case(B), key='mean_dice')print(f'\n{A}  vs  {B}')for k, v in cmp.items(): print(f'  {k}: {v}')

In [ ]:
# FDR across several comparisonspairs = [(names[i], names[j]) for i in range(len(names)) for j in range(i+1, len(names))][:6]rows = []for a_, b_ in pairs:    c = compare_arms(per_case(a_), per_case(b_), key='mean_dice')    rows.append({'a': a_, 'b': b_, 'delta': c['delta'],                 'p': (c['wilcoxon'] or {}).get('p'), 'd': c['cohens_d']})df = pd.DataFrame(rows)valid = df['p'].notna()if valid.any():    rej, adj = fdr_correct(df.loc[valid,'p'].tolist())    df.loc[valid,'p_fdr'] = adj; df.loc[valid,'significant'] = rejdf

## 2.2 --- Volume stratificationDoes performance collapse on small lesions? This is the depth layer that turns a benchmark into a finding.

In [ ]:
from glioseg.metrics import stratified_by_volumeimport numpy as npTARGET = names[0]     # <-- the run to analysepc = per_case(TARGET)strat = stratified_by_volume(pc, region='ET')display(pd.DataFrame(strat))d = [c['ET_dice'] for c in pc if not c.get('ET_gt_empty')]print(f'ET Dice {np.mean(d):.4f}  95% CI {bootstrap_ci(d)}  n={len(d)}')

## 2.3 --- Post-processing ablation (inference only, near-free)Removing components under 50 voxels is what the official BraTS metric code does. It also deletes true small lesions — which is exactly the trade-off worth measuring rather than assuming.

In [ ]:
import torchfrom glioseg.config import Configfrom glioseg.models import build_modelfrom glioseg.data import build_loadersfrom glioseg.evaluate import evaluatecfgd = json.loads((Path(runs[TARGET]['_dir'])/'config.json').read_text())cfgd.pop('out_root', None)cfg = Config(**{k: (tuple(v) if isinstance(v, list) else v)                for k, v in cfgd.items() if k in Config.__dataclass_fields__})cfg.out_root = OUTmodel = build_model(cfg).cuda()model.load_state_dict(torch.load(f"{runs[TARGET]['_dir']}/best.pt",                                 map_location='cuda', weights_only=False)['model'])_, _, test_loader, _ = build_loaders(cfg, f'{DRIVE}/results/split_brats2021.json', verbose=False)rows = []for mv in [0, 10, 50, 100]:    cfg.postproc_min_voxels = mv    agg, _ = evaluate(model, test_loader, cfg, with_ap=False)    rows.append({'min_voxels': mv, 'mean_dice': agg['mean_dice_mean'],                 'ET_dice': agg['ET_dice_mean'], 'WT_dice': agg['WT_dice_mean']})cfg.postproc_min_voxels = 0pd.DataFrame(rows)

## 2.4 --- XAI: Grad-CAM (rubric-mandatory)Grad-CAM was defined for classifiers. Segmentation has no single scalar, so we differentiate the summed channel logit over the predicted region, following Vinogradova et al., AAAI 2020. **Say this in the report** — an examiner who knows Grad-CAM will ask.Target layers must be convolutional. Transformer blocks emit `(B,N,C)` tokens with no spatial grid.

In [ ]:
import numpy as npfrom glioseg.xai import SegGradCAM, pick_target_layer, overlay_figure, occlusion_sensitivityfrom glioseg.evaluate import predict_volumebatch = next(iter(test_loader))probs, pred, gt = predict_volume(model, batch, cfg)layer = pick_target_layer(model, cfg.model)cam_fn = SegGradCAM(model, layer)crop = batch['image'][:, :, :cfg.patch_size[0], :cfg.patch_size[1], :cfg.patch_size[2]].cuda()cam = cam_fn(crop, class_idx=2)          # 2 = ET channelcam_fn.remove()s = cfg.patch_sizefig = overlay_figure(batch['image'][0,0,:s[0],:s[1],:s[2]].cpu().numpy(), cam,                     gt[2,:s[0],:s[1],:s[2]], pred[2,:s[0],:s[1],:s[2]],                     out_path=f'{DRIVE}/results/gradcam_{cfg.model}_ET.png',                     title=f'Grad-CAM, enhancing tumour, {cfg.model}')from IPython.display import Image; Image(f'{DRIVE}/results/gradcam_{cfg.model}_ET.png')

In [ ]:
# occlusion sensitivity -- hook-free cross-check. Two XAI methods agreeing is# far more convincing than one. Slow: run on 1-2 cases only.occ = occlusion_sensitivity(model, crop, class_idx=2, patch=16, stride=16)overlay_figure(batch['image'][0,0,:s[0],:s[1],:s[2]].cpu().numpy(), occ,               gt[2,:s[0],:s[1],:s[2]], pred[2,:s[0],:s[1],:s[2]],               out_path=f'{DRIVE}/results/occlusion_{cfg.model}_ET.png',               title=f'Occlusion sensitivity, ET, {cfg.model}')Image(f'{DRIVE}/results/occlusion_{cfg.model}_ET.png')

## 2.5 --- LaTeX tablesDrop the `.tex` files straight into Overleaf. **Verify every external number in the comparison table against its source before submitting**, and state whether each is validation-set or hidden-test.

In [ ]:
from glioseg.tables import all_tables, ablation_table, data_efficiency_tablearch = [r for r in runs.values() if r['name'].endswith('-arch')]abl  = [r for r in runs.values() if '-abl' in r['name']]de   = [r for r in runs.values() if '-n' in r['name']]xfer = [r for r in runs.values() if r['name'].startswith('xfer')]tabs = all_tables(arch or list(runs.values()), TEX,                  strat_rows=strat, transfer_results=xfer or None)if abl:    for f in ['loss','mod','patch']:        sub = [r for r in abl if f in r['name']]        if sub:            Path(f'{TEX}/tab_abl_{f}.tex').write_text(ablation_table(sub, f))if de:    Path(f'{TEX}/tab_data_efficiency.tex').write_text(data_efficiency_table(de))!ls -la "{TEX}"print(tabs['performance'])

In [ ]:
# bundle everything for the report!cd "{DRIVE}/results" && zip -qr "{DRIVE}/499A_report_assets.zip" tex *.png *.jsonl *.jsonprint('-> 499A_report_assets.zip on Drive')